# Triton Kernel 主线 · 第 7/10 课：稳定 Softmax 与融合

> 状态：**参考答案版**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：实现一行 softmax，解释 `-inf` padding 和数值稳定性。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：Python、PyTorch 张量、CUDA 基本线程/内存概念
- 本课在路线中的作用：softmax 需要行 max、指数和 sum 三步归约；Triton 将整行保留在片上，避免物化中间 tensor。

## 核心心智模型

### 1. 它是什么，解决什么问题

softmax 需要行 max、指数和 sum 三步归约；Triton 将整行保留在片上，避免物化中间 tensor。

### 2. 它如何工作

padding load 用 -inf，使其不影响 max 且 exp 后自然为 0；减 max 后再指数。

### 3. 正确性条件与常见误区

全 mask 行会产生 -inf-(-inf)=NaN；普通 dense 行要求 N>0，attention mask 需专门处理。

### 4. 性能与工程取舍

单 program 一行适合中等 N；超宽 softmax 需分块或专用算法。

## 具体演示

[1000,1001] 减 max 后 [-1,0]；直接 exp 会溢出。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐 padding 值。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def softmax_kernel(x, out, N: tl.constexpr, stride_m: tl.constexpr,
                   BLOCK: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    mask = cols < N
    values = tl.load(x + row * stride_m + cols, mask=mask, other=______)  # TODO: max 的单位元
    values = values - tl.max(values, axis=0)
    numer = tl.exp(values)
    probs = numer / tl.sum(numer, axis=0)
    tl.store(out + row * N + cols, probs, mask=mask)

def softmax(x):
    assert x.ndim == 2 and x.stride(1) == 1
    M, N = x.shape
    out = torch.empty_like(x)
    softmax_kernel[(M,)](x, out, N, x.stride(0), BLOCK=triton.next_power_of_2(N))
    return out

for shape in ((2, 7), (4, 100), (8, 256)):
    x = torch.randn(shape, device="cuda") * 20
    torch.testing.assert_close(softmax(x), torch.softmax(x, 1), atol=2e-4, rtol=2e-4)


### 检查方法

在 CUDA/Triton 环境运行本单元格；断言覆盖规则尺寸和非规则尾块。首次 JIT 不计入性能。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“稳定 Softmax 与融合”的工作机制。

**你的答案：**


### Q2

padding 用 0 在 logits 全负时会造成什么错误？

**你的答案：**


### Q3

如何把 causal mask 融入同一个 kernel？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考答案（仅 answer 分支）

先完成题目再核对。即使代码一致，也要能解释关键步骤，并尝试更换一个输入规模。

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def softmax_kernel(x, out, N: tl.constexpr, stride_m: tl.constexpr,
                   BLOCK: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    mask = cols < N
    values = tl.load(x + row * stride_m + cols, mask=mask, other=-float("inf"))
    values = values - tl.max(values, axis=0)
    numer = tl.exp(values)
    probs = numer / tl.sum(numer, axis=0)
    tl.store(out + row * N + cols, probs, mask=mask)

def softmax(x):
    assert x.ndim == 2 and x.stride(1) == 1
    M, N = x.shape
    out = torch.empty_like(x)
    softmax_kernel[(M,)](x, out, N, x.stride(0), BLOCK=triton.next_power_of_2(N))
    return out

for shape in ((2, 7), (4, 100), (8, 256)):
    x = torch.randn(shape, device="cuda") * 20
    torch.testing.assert_close(softmax(x), torch.softmax(x, 1), atol=2e-4, rtol=2e-4)


### Q1 参考答案

padding load 用 -inf，使其不影响 max 且 exp 后自然为 0；减 max 后再指数。

### Q2 参考答案

判断时先检查本课不变量：全 mask 行会产生 -inf-(-inf)=NaN；普通 dense 行要求 N>0，attention mask 需专门处理。  若不成立，最终数值或系统状态即使暂时正常也不可信。

### Q3 参考答案

迁移时先保证正确性，再比较代价。这里的核心取舍是：单 program 一行适合中等 N；超宽 softmax 需分块或专用算法。

## 参考资料

- [Triton Tutorials](https://triton-lang.org/main/getting-started/tutorials/)
- [Triton language API](https://triton-lang.org/main/python-api/triton.language.html)

资料用于建立事实基线；面试回答仍需用自己的语言组织。